# AgriSense — Plant Species + Disease Recognition (Colab GPU training)

Trains a species classifier (Stage 1) and one disease classifier per
species (Stage 2), evaluates them honestly (including a real-world
PlantDoc test), and exports them for the serving API.

Full design: `plant-disease-implementation-plan.md` (the spec this code
implements) and `plant-disease-full-roadmap-v2.md` (the original
roadmap) in the repo root. `ml/README.md` has the quickstart summary.

**Before opening this notebook:** push your local changes (including the
whole `ml/` folder) to GitHub — cell 1 clones the repo fresh into Colab.

## Runtime setup
**Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4)**,
then run cells top to bottom, in order.

## If Colab disconnects mid-training
Reconnect, then:
1. Re-run cells 1-3 (clone/pull + mount Drive + env check + ensure_dirs) — cheap.
2. Re-run cell 5 (materialize) — rebuilds the local symlink trees from the
   manifest on Drive in seconds; nothing was lost.
3. Re-run whichever training cell you were on (Stage 1 or Stage 2). Both
   are resumable: they pick up from the last checkpoint synced to Drive,
   they do **not** restart from scratch.

## Manual checkpoints (cells clearly marked below)
- **Cell 4**: upload your Kaggle `kaggle.json` when prompted.
- **Cell 7**: read the taxonomy "review" list it prints, fill in
  `ml/configs/taxonomy_overrides.yaml`, then re-run cell 7.
- **Cells 15-16**: conditional — only run if Phase E1's report
  (`artifacts/comparison.md`) shows the documented drop is bad enough on
  multi-object photos to warrant it, or if a species needs Phase F
  augmentation.


In [ ]:
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
service = build('drive', 'v3')
about = service.about().get(fields="user").execute()
print("Drive account:", about['user']['emailAddress'])


Drive account: ankultiwariofficial@gmail.com


In [15]:
import os
print("Is /content/drive actually a real Drive mount?:", os.path.ismount('/content/drive'))
print()
print("What's actually in /content/drive/MyDrive (if anything):")
!ls -la /content/drive/MyDrive/ 2>&1


Is /content/drive actually a real Drive mount?: True

What's actually in /content/drive/MyDrive (if anything):
total 930
drwx------ 8 root root   4096 Aug 23 06:13  AgriSense_PlantDisease
drwx------ 2 root root   4096 Feb 19  2026 'Colab Notebooks'
-rw------- 1 root root  82651 Aug 16  2024 'Grey Simple Modern Minimalist Web Developer CV Resume A4 Document (1).pdf'
drwx------ 2 root root   4096 Oct  6  2025  Plastic_dataset
-rw------- 1 root root 136683 Aug  1  2025 'Screenshot 2025-08-01 215126.png'
-rw------- 1 root root 707826 Sep 25  2025 'SIH2025-IDEA-Presentation-Format (2).pptx'
-rw------- 1 root root  11658 Oct  6  2025  Untitled4.ipynb
-rw------- 1 root root    184 Nov  5  2023 'Untitled spreadsheet.gsheet'


In [14]:
import os, shutil

mp = '/content/drive'
# Guard: if Drive were actually mounted, deleting this would nuke your Drive.
assert not os.path.ismount(mp), "Drive IS mounted — do NOT delete. Just re-run cell 2."
print("contents before:", os.listdir(mp) if os.path.isdir(mp) else "(does not exist)")
shutil.rmtree(mp, ignore_errors=True)
print("cleared. now re-run cell 2.")

AssertionError: Drive IS mounted — do NOT delete. Just re-run cell 2.

In [13]:
# Cell 1 — clone/pull the repo, install pinned Colab dependencies.
# Re-run this cell after any disconnect; it's a no-op if already cloned
# (git pull only) and pip install is idempotent.
import os

REPO_URL = "https://github.com/ankul5/AgriSense-AI-.git"
REPO_DIR = "/content/AgriSense"

if not os.path.exists(REPO_DIR):
    get_ipython().system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    get_ipython().system(f'cd {REPO_DIR} && git pull')

os.chdir(f"{REPO_DIR}/ml")
os.environ["PYTHONPATH"] = f"{REPO_DIR}/ml/src"

get_ipython().system('pip -q install -r requirements-colab.txt')
print("Repo ready at", REPO_DIR)


Already up to date.
Repo ready at /content/AgriSense


In [4]:
# Cell 2 — mount Drive, then confirm GPU/torch/ultralytics are sane.
# Hard-fails with a clear message if no GPU is attached — see
# plant-disease-implementation-plan.md section A1.
import sys
sys.path.insert(0, "/content/AgriSense/ml/src")

from agrisense_pd import config, drive_io

drive_io.mount()
config.env_report()


Mounted at /content/drive
2026-08-24 16:14:41 [INFO] drive_io: Drive mounted at /content/drive
On Colab: True
Drive root: /content/drive/MyDrive/AgriSense_PlantDisease
Local root: /content/agrisense_pd
Local disk free space: 69.6 GB
Drive free space: 7.2 GB
torch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB
CUDA version (torch build): 12.8
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
ultralytics version: 8.4.127
Environment check passed.


In [5]:
# Cell 3 — create the full Drive + local folder skeleton (idempotent).
from agrisense_pd import config

config.ensure_dirs()
print("Drive root:", config.PATHS.drive_root)
print("Local root:", config.PATHS.local_root)


Drive root: /content/drive/MyDrive/AgriSense_PlantDisease
Local root: /content/agrisense_pd


### Cell 4 — Kaggle credentials (manual step)
PlantVillage downloads via the Kaggle API. If you already uploaded
`kaggle.json` in a previous session, Drive-persisted credentials mean you
may not need to repeat this — the download script tells you plainly if
it can't find them.

In [ ]:
# Cell 4 — upload kaggle.json (skip if you've already done this and it's
# cached in this runtime — the download script will tell you if it's missing).
import os
from pathlib import Path

kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
if kaggle_json.exists():
    print("kaggle.json already present, skipping upload.")
else:
    from google.colab import files
    print("Upload your kaggle.json (from kaggle.com/settings/account -> API -> Create New Token):")
    uploaded = files.upload()
    os.makedirs(kaggle_json.parent, exist_ok=True)
    for name in uploaded:
        if name == "kaggle.json":
            Path(name).rename(kaggle_json)
    os.chmod(kaggle_json, 0o600)
    print("Saved to", kaggle_json)


kaggle.json already present, skipping upload.


### Phase A — Setup & Data Collection

In [6]:
# Cell 5 — Phase A2: download PlantVillage, Digipathos, PlantDoc.
# If Digipathos's automated fetch fails (its DOI landing page has no
# stable direct-download URL), this prints exact manual-download
# instructions — that is expected, not a bug. Follow them, then re-run.
!python -m agrisense_pd.data.download


2026-08-24 16:15:45 [INFO] download: === Processing dataset: cassava ===
2026-08-24 16:15:46 [INFO] download: cassava archive already in Drive, skipping download.
2026-08-24 16:15:46 [INFO] drive_io: Copying cassava.zip (2.4GB) from Drive to local disk...
2026-08-24 16:16:45 [INFO] drive_io: Copied cassava.zip to /content/agrisense_pd/_archives_tmp/cassava.zip
2026-08-24 16:16:45 [INFO] download: Extracting /content/agrisense_pd/_archives_tmp/cassava.zip -> /content/agrisense_pd/data/raw/cassava
2026-08-24 16:17:18 [INFO] download: Extraction complete for cassava
2026-08-24 16:17:18 [INFO] download: === Processing dataset: plantdoc ===
2026-08-24 16:17:18 [INFO] download: Cloning plantdoc from https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset.git ...
2026-08-24 16:20:28 [INFO] download: === Processing dataset: plantvillage ===
2026-08-24 16:20:28 [INFO] download: plantvillage archive already in Drive, skipping download.
2026-08-24 16:20:28 [INFO] drive_io: Copying plantv

In [ ]:
# Cell 6 — Phase A3: inspect each dataset's structure independently.
# Read artifacts/inspect_*.md (on Drive) before moving on — you should be
# able to state in one sentence how each dataset names its classes.
!python -m agrisense_pd.data.inspect_structure


### Phase B — Cleaning & Label Harmonization

In [ ]:
# Cell 7 — Phase B1: build the unified species/condition taxonomy.
# Prints a "NEEDS REVIEW" list for any source label it could not
# confidently resolve — it never guesses these.
!python -m agrisense_pd.data.taxonomy --build


### Cell 8 — manual step
If cell 7 printed a non-empty "NEEDS REVIEW" list: open
`ml/configs/taxonomy_overrides.yaml` in the file browser (or edit it
directly here), add an entry for each reviewed label, then **re-run
cell 7**. Repeat until the review list is empty.

In [ ]:
# Cell 9 — Phase B2: clean & fingerprint (dedup + quality checks).
# Checkpoints every 5000 images to Drive, so a disconnect mid-pass only
# costs the images since the last checkpoint, not the whole run.
!python -m agrisense_pd.data.clean --workers 8


In [ ]:
# Cell 10 — Phase B3: build manifests/master.csv, the source of truth
# for every step from here on. Review the printed low-data / single-condition
# lists — the low-data ones are Phase F candidates.
!python -m agrisense_pd.data.build_manifest


### Phase C — Reorganize for Two-Stage Training

In [ ]:
# Cell 11 — Phase C0: ONE split, shared by both stages, dup-group aware.
!python -m agrisense_pd.data.split


2026-08-23 10:16:27 [INFO] split: Wrote split assignments back to /content/drive/MyDrive/AgriSense_PlantDisease/manifests/master.csv
2026-08-23 10:16:28 [INFO] split: Wrote /content/drive/MyDrive/AgriSense_PlantDisease/artifacts/split_report.md

Total images split: 98354
  train: 78676 (80.0%)
  val: 9864 (10.0%)
  test: 9814 (10.0%)


In [7]:
# Cell 12 — Phase C1/C2: materialize the symlinked training trees.
# Fast (seconds) — if this takes minutes, it fell back to copying instead
# of symlinking; check the printed link mode.
!python -m agrisense_pd.data.materialize --stage 1 --stage 2


2026-08-24 16:27:14 [INFO] materialize: Stage 1 tree materialized at /content/agrisense_pd/data/stage1_species using link mode=symlink
[stage1] total images: 98354, groups: 105
[stage1] counts match manifest exactly.
2026-08-24 16:27:15 [INFO] materialize: 34 species have >=2 conditions and get a Stage 2 tree; 1 species are single-condition and are skipped (constant prediction at serve time).
2026-08-24 16:27:26 [INFO] materialize: Stage 2 trees materialized at /content/agrisense_pd/data/stage2_disease using link mode=symlink
[stage2/apple] total images: 4536, groups: 18
[stage2/apple] counts match manifest exactly.
[stage2/banana] total images: 933, groups: 21
[stage2/banana] counts match manifest exactly.
[stage2/basil] total images: 674, groups: 6
[stage2/basil] counts match manifest exactly.
[stage2/bean] total images: 647, groups: 12
[stage2/bean] counts match manifest exactly.
[stage2/bell_pepper] total images: 3047, groups: 18
[stage2/bell_pepper] counts match manifest exactly.


In [8]:
import os, glob
p = glob.glob('/content/agrisense_pd/data/stage1_species/train/wheat/*.jpg')[:3]
for f in p:
    print(os.path.exists(f), '<-', os.readlink(f) if os.path.islink(f) else 'not a link')

True <- /content/agrisense_pd/data/raw/plantwild/plantwild_v2/wheat head scab/wheat_head_scab_Baidu_0495.jpg
True <- /content/agrisense_pd/data/raw/plantwild/plantwild_v2/wheat head scab/wheat_head_scab_Baidu_0233.jpg
True <- /content/agrisense_pd/data/raw/plantwild/plantwild_v2/wheat stripe rust/wheat_stripe_rust_Baidu_0280.jpg


### Phase D — Train
**This is where the GPU does real work.** Stage 1 takes roughly 1.5-3
hours on a free-tier T4. Checkpoints sync to Drive every 2 epochs — if
this disconnects, just re-run this same cell after reconnecting (cells
1-3 first); it resumes from the last synced checkpoint.

In [10]:
# Cell 13 — Phase D1: train the Stage 1 species classifier.
!python -m agrisense_pd.train.stage1


2026-08-24 16:32:19 [INFO] drive_io: Pulled checkpoint /content/drive/MyDrive/AgriSense_PlantDisease/models/stage1/last.pt -> /content/agrisense_pd/runs/_resume/stage1_last.pt
2026-08-24 16:32:19 [INFO] train.stage1: Found existing checkpoint on Drive — resuming Stage 1 training.
Ultralytics 8.4.127 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/agrisense_pd/data/stage1_species, degrees=10.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.2, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript,

### Cell 14 — Phase D2: per-species disease classifiers
This is a resumable loop over every multi-condition species — expect to
need 2+ sessions for the full set (~20-30 species). `--max-minutes` makes
it stop cleanly between species before a session is likely to die; just
re-run the same cell to continue with whatever species are left.

In [ ]:
# Cell 14 — Phase D2: train one disease classifier per species.
# Adjust --max-minutes to fit your remaining session time; re-run this
# same cell across multiple sessions until the summary shows 0 remaining.
!python -m agrisense_pd.train.stage2 --max-minutes 300


### Phase E — Test for Real-World Accuracy

In [ ]:
# Cell 15 — Phase E0: end-to-end evaluation on OUR OWN held-out test split.
# This is the clean-data baseline that Phase E1's PlantDoc number gets
# compared against.
!python -m agrisense_pd.eval.evaluate_holdout


In [ ]:
# Cell 16 — Phase E1: real-world generalization test on PlantDoc.
# EXPECT a large drop vs cell 15's numbers — a clean-lab top-1 in the high
# 90s and a PlantDoc end-to-end result in the 40-70% range is the normal,
# documented outcome for PlantVillage-trained models, not a bug. Read
# artifacts/comparison.md and artifacts/plantdoc_report.md afterward.
!python -m agrisense_pd.eval.plantdoc_eval


### Cell 17 — Phase E2 (CONDITIONAL): leaf detector pre-step
Only run this if cell 16's `multi_object` metrics are materially worse
(roughly >10 points of strict_e2e) than its `single_object` metrics. If
they're close, skip straight to cell 19 (export) — a detector you don't
need only adds latency and a failure mode.

In [ ]:
# Cell 17 — Phase E2: build + evaluate the leaf detector (only if needed).
!python -m agrisense_pd.detect.plantdoc_to_yolo
!python -m agrisense_pd.detect.train_detector
# Check artifacts/phase_e2_detector_report.md's printed decision before
# treating the detector as part of the shipped pipeline.


### Cell 18 — Phase F (CONDITIONAL): fix flagged species
For each species flagged in cell 14's summary (test top-1 < 0.85) or in
`artifacts/class_report.md`'s low-data list: add an override under
`overrides:` in `ml/configs/train_stage2.yaml`, then retrain with the
aggressive augmentation profile. This writes to a **separate**
`<species>__aug` run — it does not overwrite the baseline.

In [ ]:
# Cell 18 — Phase F: retrain a flagged species with aggressive augmentation.
# Edit SPECIES_TO_FIX below, and add an `overrides:` entry for it in
# ml/configs/train_stage2.yaml first (see that file's example comment).
SPECIES_TO_FIX = "REPLACE_ME"  # e.g. "corn"

!python -m agrisense_pd.train.stage2 --species {SPECIES_TO_FIX} --augment-profile aggressive

# Compare artifacts/stage2_report.md's baseline row against the new
# <species>__aug run (val/test top-1), then re-run cell 16 restricted to
# this species if you want the PlantDoc-level comparison too. Promote
# (copy the __aug best.pt over models/stage2/<species>/best.pt on Drive)
# ONLY if it improves the PlantDoc/real-world number, not just validation —
# record the decision in artifacts/phase_f_<species>.md either way.


### Phase G — Export for Deployment

In [ ]:
# Cell 19 — Phase G: export to ONNX, verify parity, write the serving registry.
!python -m agrisense_pd.export.to_onnx
!python -m agrisense_pd.export.verify_onnx
!python -m agrisense_pd.export.registry


### Cell 20 — package the exported models for download
Downloads a zip of `Drive/AgriSense_PlantDisease/exported/` (registry.json
+ every ONNX file) so you can place it at `serving/models/` locally. See
`serving/DEPLOY.md` for the rest of the path to a live API.

In [ ]:
# Cell 20 — zip exported/ and download it.
from agrisense_pd.config import PATHS
import shutil

zip_path = shutil.make_archive("/content/agrisense_exported", "zip", str(PATHS.exported))
print("Created", zip_path)

from google.colab import files
files.download(zip_path)
